In [1]:
!pip install -q condacolab
import condacolab
condacolab.install_from_url(
    "https://github.com/conda-forge/miniforge/releases/download/25.3.1-0/Miniforge3-Linux-x86_64.sh"
)

⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/25.3.1-0/Miniforge3-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:11
🔁 Restarting kernel...


In [1]:
!sed -i 's/python=3.12/python=3.11/' /usr/local/conda-meta/pinned || true

In [ ]:
!mamba install -y -c conda-forge openff-toolkit

In [ ]:
!pip uninstall -y openff-nagl pytorch-lightning torchmetrics torchvision torch

In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
from rdkit import Chem
from openff.toolkit.topology import Molecule, Topology
from openff.toolkit.typing.engines.smirnoff import ForceField


In [2]:
from pathlib import Path
import re




In [3]:
def safe_name(name, default):
    if not name:
        return default
    name = re.sub(r"[^\\w.\\-]+", "_", name.strip())
    name = name.strip("._")
    return name or default


def get_mol_name(rdmol, i):
    name = ""
    if rdmol.HasProp("Molecule Name"):
        name = rdmol.GetProp("Molecule Name").strip()
    return safe_name(name, f"ligand_{i}")


def parameterize_sdf(sdf_path, output_dir, ff_name="openff-2.2.0.offxml"):
    sdf_path = Path(sdf_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True, parents=True)

    ff = ForceField(ff_name)
    supplier = Chem.SDMolSupplier(str(sdf_path), removeHs=False)

    for i, rdmol in enumerate(supplier, start=1):
        if rdmol is None:
            continue

        name = get_mol_name(rdmol, i)
        mol_dir = output_dir / name

        j = 2
        while mol_dir.exists():
            mol_dir = output_dir / f"{name}_{j}"
            j += 1

        mol_dir.mkdir()

        offmol = Molecule.from_rdkit(
            rdmol,
            allow_undefined_stereo=True,
            hydrogens_are_explicit=True,
        )
        offmol.name = mol_dir.name

        topology = Topology.from_molecules([offmol])
        interchange = ff.create_interchange(topology)

        interchange.to_pdb(str(mol_dir / "ligand.pdb"))
        #interchange.to_gro(str(mol_dir / "ligand.gro"))
        interchange.to_top(str(mol_dir / "topol.top"))

        print(f"Done: {mol_dir.name}")


In [ ]:
sdf_path = "/content/drive/MyDrive/example_open_ff.sdf"
output_dir = "openff_results"

parameterize_sdf(sdf_path, output_dir)


In [ ]:
!zip -r results.zip openff_results

  adding: openff_results/ (stored 0%)
  adding: openff_results/ligand_11/ (stored 0%)
  adding: openff_results/ligand_11/topol.top (deflated 89%)
  adding: openff_results/ligand_11/ligand.pdb (deflated 76%)
  adding: openff_results/ligand_7/ (stored 0%)
  adding: openff_results/ligand_7/topol.top (deflated 89%)
  adding: openff_results/ligand_7/ligand.pdb (deflated 76%)
  adding: openff_results/ligand_5/ (stored 0%)
  adding: openff_results/ligand_5/topol.top (deflated 89%)
  adding: openff_results/ligand_5/ligand.pdb (deflated 76%)
  adding: openff_results/ligand_6/ (stored 0%)
  adding: openff_results/ligand_6/topol.top (deflated 89%)
  adding: openff_results/ligand_6/ligand.pdb (deflated 76%)
  adding: openff_results/ligand_12/ (stored 0%)
  adding: openff_results/ligand_12/topol.top (deflated 89%)
  adding: openff_results/ligand_12/ligand.pdb (deflated 76%)
  adding: openff_results/ligand_10/ (stored 0%)
  adding: openff_results/ligand_10/topol.top (deflated 89%)
  adding: openff_r

In [ ]:
from google.colab import files
files.download("results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>